## Project Note: Computational Constraints & Architectural Adaptations

Due to localized hardware limitations and restricted computational overhead, this model was developed and validated using a curated subset of the primary dataset. To compensate for the reduced sample size and maintain high generalization performance, the following architectural modifications were implemented:

Temporal Downsampling: Optimized frame selection via uniform interval sampling to maintain sequence context while reducing memory footprint.

Transfer Learning Integration: Leveraged pre-trained spatial feature extractors to minimize the need for massive data volumes.

Regularization Tuning: Increased Dropout and Weight Decay parameters to mitigate overfitting risks inherent in small-batch training.

In [1]:
from google.colab import files
files.upload()

# 2. Set up the Kaggle API environment
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Smaller but similar dataset (~3.5GB)
!kaggle datasets download -d mohamedmustafa/real-life-violence-situations-dataset
!unzip -q real-life-violence-situations-dataset.zip -d ./mini_dataset

Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/mohamedmustafa/real-life-violence-situations-dataset
License(s): copyright-authors
 99% 3.57G/3.58G [00:23<00:00, 228MB/s]
100% 3.58G/3.58G [00:23<00:00, 163MB/s]


In [2]:
# 1. Unzip the dataset
!unzip -q real-life-violence-situations-dataset.zip -d ./dataset

# 2. Delete the zip to free up 3.5GB of space
!rm real-life-violence-situations-dataset.zip

import os
print(os.listdir('./dataset/Real Life Violence Dataset'))

['Violence', 'NonViolence']


In [3]:
import cv2
import numpy as np

def get_frames_from_video(video_path, target_frames=20, img_size=128):
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    interval = max(1, total_frames // target_frames)

    frames = []

    for i in range(target_frames):
        # Set the 'camera' to the specific frame index
        cap.set(cv2.CAP_PROP_POS_FRAMES, i * interval)
        ret, frame = cap.read()

        if ret:
            # Resize and Normalize (0 to 1)
            frame = cv2.resize(frame, (img_size, img_size))
            frame = frame / 255.0
            frames.append(frame)
        else:
            # If a frame fails to load, append a black frame to keep the shape consistent
            frames.append(np.zeros((img_size, img_size, 3)))

    cap.release()
    return np.array(frames)

In [4]:
import numpy as np
import cv2

# Define paths based on the unzip folder
BASE_DIR = './dataset/Real Life Violence Dataset/'
categories = ['Violence', 'NonViolence']

X = []
y = []

# Loop through each category and grab a subset
for label, cat in enumerate(categories):
    folder_path = os.path.join(BASE_DIR, cat)
    video_files = os.listdir(folder_path)[:50] # Subset: 50 videos per class

    print(f"Processing {cat}...")
    for vid in video_files:
        path = os.path.join(folder_path, vid)
        # Calling your function!
        frames = get_frames_from_video(path, target_frames=20)
        X.append(frames)
        y.append(label) # 0 for Violence, 1 for Non-Violence

# Convert to numpy arrays
X = np.array(X)
y = np.array(y)

print(f"Data shape: {X.shape}") # Should be (100, 20, 128, 128, 3)

Processing Violence...
Processing NonViolence...
Data shape: (100, 20, 128, 128, 3)


In [5]:
from sklearn.model_selection import train_test_split

# Shuffle and split: 80% for training, 20% for validation
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, shuffle=True, random_state=42)

print(f"Training samples: {len(X_train)}")
print(f"Validation samples: {len(X_val)}")

Training samples: 80
Validation samples: 20


In [6]:
from tensorflow.keras import layers, models

def build_bilstm_model(seq_len=20, img_size=128):
    model = models.Sequential([
        # We apply the same Conv layers to all 20 frames
        layers.TimeDistributed(layers.Conv2D(16, (3, 3), activation='relu'),
                               input_shape=(seq_len, img_size, img_size, 3)),
        layers.TimeDistributed(layers.MaxPooling2D((2, 2))),

        layers.TimeDistributed(layers.Conv2D(32, (3, 3), activation='relu')),
        layers.TimeDistributed(layers.MaxPooling2D((2, 2))),

        layers.TimeDistributed(layers.Flatten()),
        # It looks at the sequence forwards and backwards
        layers.Bidirectional(layers.LSTM(64, return_sequences=False)),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.5), # Helps prevent overfitting on our small subset
        layers.Dense(1, activation='sigmoid') # 0 = Violence, 1 = Non-Violence
    ])

    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

model = build_bilstm_model()
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/wrapper.py:27: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ time_distributed                │ (None, 20, 126, 126,   │           448 │
│ (TimeDistributed)               │ 16)                    │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_1              │ (None, 20, 63, 63, 16) │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_2              │ (None, 20, 61, 61, 32) │         4,640 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_3              │ (None, 20, 30, 30, 32) │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_4              │ (None, 20, 28800)      │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 128)            │    14,778,880 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,792,289 (56.43 MB)

 Trainable params: 14,792,289 (56.43 MB)

 Non-trainable params: 0 (0.00 B)

In [7]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=4, # Small batch size because video data is heavy
    verbose=1
)

Epoch 1/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 128s 6s/step - accuracy: 0.5382 - loss: 0.9126 - val_accuracy: 0.4000 - val_loss: 0.7144
Epoch 2/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 146s 6s/step - accuracy: 0.4473 - loss: 0.7154 - val_accuracy: 0.7500 - val_loss: 0.6116
Epoch 3/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 140s 6s/step - accuracy: 0.5595 - loss: 0.6936 - val_accuracy: 0.5500 - val_loss: 0.6592
Epoch 4/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 122s 6s/step - accuracy: 0.7084 - loss: 0.5444 - val_accuracy: 0.4500 - val_loss: 0.7913
Epoch 5/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 149s 6s/step - accuracy: 0.6846 - loss: 0.5270 - val_accuracy: 0.5500 - val_loss: 0.7653
Epoch 6/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 148s 7s/step - accuracy: 0.8707 - loss: 0.3017 - val_accuracy: 0.6000 - val_loss: 0.6619
Epoch 7/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 127s 6s/step - accuracy: 0.9575 - loss: 0.2154 - val_accuracy: 0.6000 - val_loss: 0.7216
Epoch 8/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 143s 6s/step - accuracy: 1.0000 - loss: 0.0976 - val_accuracy: 0.6500 - v

In [8]:
from google.colab import files

model.save('my_bilstm_model1.h5')
files.download('my_bilstm_model1.h5')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>